In [11]:
import pandas as pd
import numpy as np

housing_df = pd.read_csv('numeric_prop_census_isd.csv')


#changed year built to years since
def years_since(x):
  return 2026-x
yr_built_correct = housing_df['rs_prop_yr_built'].map(years_since)
print(yr_built_correct)
housing_df.insert(1, 'rs_prop_yrs_since_built',yr_built_correct)
housing_df = housing_df.drop(columns=['rs_prop_yr_built'])

#first tried fillna(0) but had bad accuracy-- replaced w mean of columns, didn't change much
#for column_name in housing_df:
  #housing_df[column_name].fillna(housing_df[column_name].mean(),inplace=True)
 
  
housing_df = housing_df.fillna(housing_df.mean())
print(housing_df.head)


#outcome column: violation_count


0         301
1         301
2         236
3         316
4         227
         ... 
315288      9
315289     22
315290     20
315291     16
315292     25
Name: rs_prop_yr_built, Length: 315293, dtype: int64
<bound method NDFrame.head of         rs_prop_sam_id  rs_prop_yrs_since_built  rs_prop_yr_remod  \
0               407453                      301       2010.000000   
1               407452                      301       2010.000000   
2                62318                      236          0.000000   
3               133181                      316          0.000000   
4                73295                      227       2014.000000   
...                ...                      ...               ...   
315288          186426                        9          0.000000   
315289           80028                       22       1523.764783   
315290           27664                       20       1523.764783   
315291          341536                       16       1523.764783   
3152

In [12]:
housing_df = housing_df.drop(columns=['json_bldg_mhl_housing_5year','json_mhl_housing_5year','rs_prop_yr_remod'])


In [13]:
#total split: 64% of data to train each classification and regression, 16% to test each, and 20% to test the two-step model
X = housing_df.drop(columns=['violation_count'])
y = housing_df['violation_count']
# from sklearn.preprocessing import minmax_scale
# X = minmax_scale(X)
print(X.shape)
X = (X-X.min(axis=0))/(X.max(axis=0)-X.min(axis=0))
print(X.shape)

from sklearn.model_selection import train_test_split
X_train_total, X_test_2step, y_train_total, y_test_2step = train_test_split(X, y, test_size=0.2, random_state=42)



X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_train_total, y_train_total, test_size = 0.2, random_state=42)

#below are the training/testing data to use for classifier since we want to have it train on just 0's and 1's

X_train_cla = X_train_reg.copy()

X_test_cla = X_test_reg.copy()

y_train_cla = y_train_reg.copy()

y_test_cla = y_test_reg.copy()

#TODO drop rows with no violations for regression - get indices from y where violations are not 0 then slice X using those indices
#X_train_reg = X_train_reg[X_train_reg['violat']]


#change classifier data to only 0 or 1
def binary(x):
  if(x==0):
    return 0
  else:
    return 1

print(y_train_cla.head)
y_train_cla = y_train_cla.map(binary)
y_test_cla = y_test_cla.map(binary)

print(y_train_cla.head)


#X_train_total.shape, X_test_2step.shape, X_train_one.shape, X_test_one.shape

(315293, 48)
(315293, 48)
<bound method NDFrame.head of 253205    0
116626    0
292726    1
51576     4
153916    0
         ..
81245     0
175956    0
141474    2
239801    0
198931    7
Name: violation_count, Length: 201787, dtype: int64>
<bound method NDFrame.head of 253205    0
116626    0
292726    1
51576     1
153916    0
         ..
81245     0
175956    0
141474    1
239801    0
198931    1
Name: violation_count, Length: 201787, dtype: int64>


In [14]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train_cla, y_train_cla)
y_pred_cla = log_reg.predict(X_test_cla)

In [15]:
from sklearn import metrics
cnf_matrix = metrics.confusion_matrix(y_test_cla, y_pred_cla)
cnf_matrix

array([[24066,  5995],
       [11318,  9068]])

In [16]:
from sklearn.metrics import classification_report
print(classification_report(y_test_cla, y_pred_cla))

              precision    recall  f1-score   support

           0       0.68      0.80      0.74     30061
           1       0.60      0.44      0.51     20386

    accuracy                           0.66     50447
   macro avg       0.64      0.62      0.62     50447
weighted avg       0.65      0.66      0.64     50447



In [17]:
from sklearn.linear_model import LinearRegression

# Filter regression data to only include rows with violations
violation_indices_train = y_train_reg[y_train_reg > 0].index
X_train_reg_filtered = X_train_reg.loc[violation_indices_train]
y_train_reg_filtered = y_train_reg.loc[violation_indices_train]

violation_indices_test = y_test_reg[y_test_reg > 0].index
X_test_reg_filtered = X_test_reg.loc[violation_indices_test]
y_test_reg_filtered = y_test_reg.loc[violation_indices_test]

lin_reg = LinearRegression()
lin_reg.fit(X_train_reg_filtered, y_train_reg_filtered)
y_pred_reg = lin_reg.predict(X_test_reg_filtered)


from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from math import sqrt

rmse = sqrt(mean_squared_error(y_test_reg_filtered, y_pred_reg))
r2 = r2_score(y_test_reg_filtered, y_pred_reg)
mae = mean_absolute_error(y_test_reg_filtered, y_pred_reg)

print(f'RMSE: {rmse}')
print(f'R^2: {r2}')
print(f'MAE: {mae}')

RMSE: 3.308508556712606
R^2: 0.15001257276781288
MAE: 2.116269305779475


In [18]:
# two step model: use classifier to predict if there will be a violation, then use regression to predict how many if classifier predicts there will be a violation
#fit on leftover 20% of data that we haven't used for either model yet
log_reg.fit(X_train_total, y_train_total.map(binary))

# Predict on training data to identify predicted positives
train_pred_cla = log_reg.predict(X_train_total)

# Filter training data to only predicted positives for regression training
X_train_reg_filtered = X_train_total[train_pred_cla == 1]
y_train_reg_filtered = y_train_total[train_pred_cla == 1]

lin_reg.fit(X_train_reg_filtered, y_train_reg_filtered)

y_pred_2step_cla = log_reg.predict(X_test_2step)
y_pred_2step_reg = lin_reg.predict(X_test_2step)
# evaluate two step model
y_pred_2step_final = []
for i in range(len(y_pred_2step_cla)):
  if y_pred_2step_cla[i] == 0:
    y_pred_2step_final.append(0)
  else:
    y_pred_2step_final.append(y_pred_2step_reg[i])


rmse_2step = sqrt(mean_squared_error(y_test_2step, y_pred_2step_final))
r_squared_2step = r2_score(y_test_2step, y_pred_2step_final)
mae_2step = mean_absolute_error(y_test_2step, y_pred_2step_final)

print(f'Two-step model RMSE: {rmse_2step}')
print(f'Two-step model R^2: {r_squared_2step}')
print(f'Two-step model MAE: {mae_2step}')

Two-step model RMSE: 2.6245600762389287
Two-step model R^2: 0.07933437462170212
Two-step model MAE: 1.2460521572218315


In [19]:
#eval two-step with 20% left out, still with class 1 
log_reg.fit(X_train_total, y_train_total.map(binary))
lin_reg.fit(X_train_total, y_train_total)
y_pred_2step_cla = log_reg.predict(X_test_2step)
y_pred_2step_reg = lin_reg.predict(X_test_2step)

mae_2step = mean_absolute_error(y_test_2step, y_pred_2step_reg)
rmse_2step = sqrt(mean_squared_error(y_test_2step, y_pred_2step_reg))
r_squared_2step = r2_score(y_test_2step, y_pred_2step_reg)

print(f'Two-step model RMSE: {rmse_2step}')
print(f'Two-step model MAE: {mae_2step}')
print(f'Two-step model R-squared: {r_squared_2step}')


Two-step model RMSE: 2.5722275119455107
Two-step model MAE: 1.4543673910662294
Two-step model R-squared: 0.11568365969914896


In [33]:
class_coef = log_reg.coef_
reg_coef = lin_reg.coef_
# print(class_coef[0].shape)
# print(reg_coef.shape)
coef = pd.DataFrame([class_coef[0], reg_coef], index=["Classification Coef", "Regression Coef"], columns=X.columns)
coef = coef.transpose()

In [36]:
coef["classcoefabs"] = coef["Classification Coef"].abs()
coef.sort_values(by="classcoefabs", ascending=False, inplace=True)
coef.drop(columns=["classcoefabs"], inplace=True)
coef

,Classification Coef,Regression Coef
Population_Sex_female_share,-2.571485,-2.387583e+00
Population_Age_share_55_plus,2.563567,9.729838e+00
rs_prop_living_area,2.378479,5.156123e+00
Housing_Units_total,1.994119,2.911877e+00
Population_ChildrenByAge_under5,-1.597369,-2.456346e+00
Housing_LivingArrangements_group_quarters,1.507682,1.802695e+07
Population_Age_0_to_9,1.397202,-7.926808e+06
Population_Age_65_plus,-1.323808,-1.322581e+07
LU_R1,-1.314327,-5.413237e-01
Housing_LivingArrangements_household_pop,-1.246852,2.834815e+07


In [ ]:
coef["regcoefabs"] = coef["Regression Coef"].abs()
coef.sort_values(by="regcoefabs", ascending=False, inplace=True)
coef.drop(columns=["regcoefabs"], inplace=True)
coef

,Classification Coef,Regression Coef
Population_Sex_total_pop,-0.288062,3.980665e+07
Population_Age_20_to_34,-0.595387,-3.930129e+07
Housing_LivingArrangements_household_pop,-1.246852,2.834815e+07
Population_Age_10_to_19,-0.327856,-1.941219e+07
Housing_LivingArrangements_group_quarters,1.507682,1.802695e+07
Population_Age_35_to_54,0.692791,-1.677292e+07
Population_Age_65_plus,-1.323808,-1.322581e+07
Population_Age_55_to_64,0.500390,-9.878156e+06
Population_Age_0_to_9,1.397202,-7.926808e+06
Population_Race_share_hispanic_latino,0.286084,1.147858e+01


: 